# PREP-03 Frozen weights, adapters, and the boundary (Day 3 words)

**Time:** 40–55 minutes &nbsp;|&nbsp; **Maps to:** T1-11 … T1-14

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/prep/PREP-03_frozen_and_boundary.ipynb)

Or: **File → Upload notebook** at [colab.research.google.com](https://colab.research.google.com/) and choose `PREP-03_frozen_and_boundary.ipynb`.


**How to use this notebook (no coding)**

1. **Runtime → Change runtime type → CPU** (not GPU). You do not need a graphics card.
2. Click ▶ on each cell from the top, or **Runtime → Run all**.
3. Read the note *under* a cell after it runs.
4. Change **only** lines marked `# [CHANGE]` — usually one word inside quotes `"..."`. Do not delete the quotes.
5. Nothing from work goes in a cell. CivicDesk names are fiction.

You are not learning Python. You are practising the same click-and-read path that no-code people use on Day 1.


## Why this night exists

Day 3 you **inspect** a tiny training file and a pretend **adapter**. You do **not** train. People say “fine-tune” when they mean “make it ours.” Usually that is a small extra file, not a new OLMo.

```
  tokenizer → embedding → transformer blocks → next-token head
                              ▲
                              └── a LoRA adapter may sit here as a thin extra file
                                  The base stays FROZEN. Rollback = stop using the file.
```

Two clocks:

| | **Burst** | **Standing** |
|---|---|---|
| What | A training / LoRA **job** | The chat **API** that stays up |
| When | Hours, then stop the extra GPU | All day (`:8000` / `:8001` in the room) |
| Fail | Buying “one box for AI” with no split | Assuming a LoRA replaces hosting |

An adapter is **not RAG** and **not the register**. Circulars still need retrieval. Serials still need a table.

**Technical staff:** [Hugging Face model cards](https://huggingface.co/docs/hub/model-cards) and [Hub security](https://huggingface.co/docs/hub/security) — official publisher vs a community fork; pin a revision. The course already pins OLMo and Phi.

**IT / policy:** skim [OWASP Top 10 for LLM Applications](https://genai.owasp.org/llm-top-10/) (excessive agency, sensitive disclosure) and the [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework) overview. T1-13 is the classroom allowlist; AUX-04 is a DPIA sketch, not a filed assessment.


## 1. Look at a training row (SFT)

**SFT** = supervised fine-tune: instruction + input + the output you wish the model had said. **JSONL** = one labelled example per line. **Held-out** = examples you do **not** train on, so you can see if the phrasing generalises. **Canary** = planted PII; it must not enter a real train split.


In [ ]:
rows = [
    {
        "id": "FT-01",
        "split": "train",
        "canary": False,
        "input": "I completed a two-year diploma in library studies in 2019.",
        "output": "Human review required. Library studies is not automatic C2 equivalent. Do not hire.",
    },
    {
        "id": "FT-08",
        "split": "train",
        "canary": True,
        "input": "Applicant Maya Reid, National ID 999-00-1111, seeks the records post.",
        "output": "REFUSE. Canary PII. Stop.",
    },
    {
        "id": "FT-10",
        "split": "train",
        "canary": False,
        "input": "Which printers are out of warranty?",
        "output": "No asset table in this row. I will not invent serials. Query the register.",
    },
]

print("id     split    canary  input")
for r in rows:
    print(f"{r['id']:6} {r['split']:8} {str(r['canary']):6} {r['input'][:70]}")
print()
print("Delete before a real job:", [r["id"] for r in rows if r["canary"]])
print("The base weights stay FROZEN in the classroom. This is a script, not a hire.")


### Teaching note

FT-01 is a fair **criteria** example (Labour/Health shape): quote, flag, do not hire. FT-08 is poison if it is trained on. FT-10 reminds you the adapter is not GLPI.


## 2. Base vs a pretend adapter


In [ ]:
def base_reply(user):
    return "[frozen base] A general CivicDesk-sounding answer. May miss the C2 equivalent clause. Still must not hire."


def adapter_reply(user):
    u = user.lower()
    if "999-00-1111" in u or "maya reid" in u:
        return "[adapter] REFUSE. Canary PII. Stop."
    if "library" in u:
        return "[adapter] Human review. Library studies is not automatic C2 equivalent. Do not hire."
    if "password" in u:
        return "[adapter] I will not reset a password. Draft ticket fields only."
    if "printer" in u or "warranty" in u:
        return "[adapter] No register table in the prompt. I will not invent serials."
    if "award" in u:
        return "[adapter] I must not award. Human review."
    return "[adapter] I am not a database. Human review for consequential acts."


prompts = [
    "I completed a diploma in library studies. Screen against C2. Do not hire.",
    "Reset my network password now.",
    "Which printers are out of warranty?",
    "Applicant Maya Reid, National ID 999-00-1111. Summarise.",
]
for p in prompts:
    print("=" * 60)
    print("USER:", p)
    print("BASE   :", base_reply(p))
    print("ADAPTER:", adapter_reply(p))


The adapter can make **phrasing** more reliable. It cannot replace HB-VAC-C2 if the rule changes (still RAG). It cannot replace the warranty table (still **integration**). If it starts emitting serials, it is treating training text as a database — fail.

**Licence inheritance (extra):** the adapter is licensed as the **base model** plus **every row** you poured in. A CPO judgment dump is not CivicDesk fiction. **Rollback:** disable the file; OLMo is unchanged.


## 3. Saying vs doing — bounded agency

Day 1–2 the model **returned text**. An **agent** may also **call tools** (reset an account, write GLPI, scrape the web). **Bounded agency** means tools are **off** unless they are on an **allowlist**, with a **human** for consequential acts.

```
  ALLOWED (saying)              BANNED (doing)
  ─────────────────             ──────────────────
  draft a ticket                reset a password
  quote a handbook span         write GLPI
  say Human review required     award a contract
                                scrape the web for negotiation
                                hire / publish a grade
```


In [ ]:
def tool_gate(action):
    table = {
        "draft_ticket": "ALLOWED. A human still files it.",
        "quote_handbook": "ALLOWED. If the paragraph is in the prompt.",
        "reset_password": "BANNED. Identity check is a human tool.",
        "write_glpi": "BANNED. No silent GLPI change.",
        "award_contract": "BANNED. The model does not award.",
        "scrape_web_for_negotiation": "BANNED. Not an open scrape from chat.",
    }
    return table.get(action, "Unknown. If it changes a live system, treat as banned until listed.")


for a in [
    "draft_ticket",
    "quote_handbook",
    "reset_password",
    "write_glpi",
    "award_contract",
    "scrape_web_for_negotiation",
    "grant_ad_rights",
]:
    print(f"{a:28}  {tool_gate(a)}")


Unknown → banned until listed. That is the default for a ministry desk. Day 3 T1-13 is this table with Trade / CPO / Defence names on it.


## 4. Arrival checklist

You are ready for Tier 1 Day 1 if you can answer **yes** to these. Change a `NO` to `YES` only when you mean it.


In [ ]:
# [CHANGE] YES or NO
CHECK = {
    "I can click Run on a Colab / Jupyter cell.": "YES",
    "I will not paste ministry files into a cell.": "YES",
    "A language model continues text; it is not the register.": "YES",
    "I can explain system vs user in one sentence.": "YES",
    "If the handbook is missing I expect 'I cannot cite', not Circular 99.": "YES",
    "A search hit is not permission to show the paragraph.": "YES",
    "Chat may draft a ticket; it may not reset a password or award.": "YES",
    "Day 3 inspects an adapter; we do not train in the room.": "YES",
}
for k, v in CHECK.items():
    flag = "ready" if v.strip().upper() == "YES" else "revisit that notebook"
    print(f"[{v.strip()}] {k}  → {flag}")
print()
print("Bring curiosity, not a USB of EDMS exports. See you on Day 1.")


## Done

Print the [prep quick card](https://github.com/justkrismanohar/fine-tuning-build/blob/main/prep/prep_quick_card.md) if you want a one-pager. Auxiliary workshops after Tier 1 (`course/auxiliary/`) are optional and still no-code.

You do not need to reopen these notebooks in the room. The Day 1–3 labs will re-teach every idea with the real CivicDesk helpers and, where a GPU exists, live OLMo/Phi.
